## Verificar Filtragem de Domínios com AWS Network Firewall

Este notebook verifica se o AWS Network Firewall implantado por `agentcore-browser-firewall.yaml` filtra corretamente domínios para sessões do AgentCore Browser.

O firewall aplica três categorias:
- **Lista de permissões** — domínios explicitamente permitidos (ex.: `example.com`, `github.com`)
- **Lista de bloqueios** — domínios explicitamente bloqueados (ex.: `facebook.com`, `twitter.com`)
- **Bloqueio padrão** — qualquer domínio que não esteja em nenhuma das listas é bloqueado

### Pré-requisitos

1. Implantar a stack CloudFormation `agentcore-browser-firewall.yaml`
2. Instalar dependências e **reiniciar seu kernel**:

In [ ]:
!pip install -qU -r requirements.txt

### 1. Configuração

Obtenha o ID do Browser das saídas da stack CloudFormation e inicialize o cliente.

In [ ]:
import boto3
from urllib.parse import urlparse
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

session = boto3.Session()
REGION = session.region_name
browser_client = boto3.client("bedrock-agentcore")

# Obter BROWSER_ID das saídas do CloudFormation
cfn = boto3.client("cloudformation")
outputs = cfn.describe_stacks(StackName="agentcore-browser-firewall")["Stacks"][0]["Outputs"]
BROWSER_ID = next(o["OutputValue"] for o in outputs if o["OutputKey"] == "BrowserToolCustomOutput")
print(f"ID do Browser: {BROWSER_ID}")

### 2. Iniciar uma sessão do navegador

Inicie uma sessão e construa a URL WebSocket assinada com SigV4 para o Playwright conectar.

In [ ]:
response = browser_client.start_browser_session(browserIdentifier=BROWSER_ID)
session_id = response["sessionId"]
ws_url = f"wss://bedrock-agentcore.{REGION}.amazonaws.com/browser-streams/{BROWSER_ID}/sessions/{session_id}/automation"
print(f"ID da Sessão: {session_id}")

# Assinar a URL WebSocket com SigV4
credentials = session.get_credentials()
https_url = ws_url.replace("wss://", "https://")
parsed = urlparse(https_url)
request = AWSRequest(method="GET", url=https_url, headers={"host": parsed.netloc})
SigV4Auth(credentials, "bedrock-agentcore", REGION).add_auth(request)
headers = {k: v for k, v in request.headers.items()}

### 3. Executar verificação de filtragem de domínios

Conecte via Playwright e tente navegar para domínios em cada categoria.

As URLs de teste abaixo correspondem aos parâmetros padrão `AllowedDomains` e `DeniedDomains` no template CloudFormation. Se você personalizou esses parâmetros, atualize as URLs de acordo.

In [ ]:
from playwright.async_api import async_playwright

# (url, categoria, deve_permitir)
tests = [
    ("https://example.com", "LISTA DE PERMISSÕES", True),
    ("https://github.com", "LISTA DE PERMISSÕES", True),
    ("https://wikipedia.org", "LISTA DE PERMISSÕES", True),
    ("https://facebook.com", "LISTA DE BLOQUEIOS", False),
    ("https://twitter.com", "LISTA DE BLOQUEIOS", False),
    ("https://randomsite12345.com", "NÃO LISTADO", False),
]

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(ws_url, headers=headers)
    page = browser.contexts[0].pages[0] if browser.contexts else await browser.new_context().new_page()

    print("=" * 60)
    print("RESULTADOS DA VERIFICAÇÃO DE FILTRAGEM DE DOMÍNIOS")
    print("=" * 60)

    results = []
    for url, category, should_allow in tests:
        try:
            resp = await page.goto(url, timeout=10000, wait_until="domcontentloaded")
            allowed = resp is not None and resp.status < 400
            passed = allowed == should_allow
            status_str = f"HTTP {resp.status}" if resp else "Sem resposta"
            result = "PASSOU" if passed else "FALHOU"
            label = "Permitido" if allowed else "Bloqueado"
            print(f"{result}: {url} ({category}) - {label} [{status_str}]")
            results.append(passed)
        except Exception as e:
            passed = not should_allow
            result = "PASSOU" if passed else "FALHOU"
            print(f"{result}: {url} ({category}) - Bloqueado ({type(e).__name__})")
            results.append(passed)

    print("=" * 60)
    print(f"Resultados: {sum(results)}/{len(results)} testes passaram")
    await browser.close()

### 4. Parar a sessão

In [ ]:
browser_client.stop_browser_session(browserIdentifier=BROWSER_ID, sessionId=session_id)
print(f"Sessão {session_id} parada")

### Solução de Problemas

Se os testes não se comportarem como esperado, verifique os logs do Network Firewall no CloudWatch:

```
/aws/network-firewall/agentcore-browser-firewall/alert
/aws/network-firewall/agentcore-browser-firewall/flow
```

Os logs de alerta mostram quais regras corresponderam. Os logs de fluxo mostram todo o tráfego através do firewall, úteis para depurar domínios que são inesperadamente bloqueados ou permitidos.